In [ ]:
import requests
from bs4 import BeautifulSoup
import textwrap
import time


In [ ]:
# Function to fetch HTML content from a URL with retry mechanism
def fetch_html(url, retries=3):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}
    attempt = 0
    while attempt < retries:
        try:
            response = requests.get(url, headers=headers, timeout=20)  # Increased timeout and added User-Agent
            response.raise_for_status()  # Check for request errors
            return response.content
        except requests.exceptions.RequestException as e:
            print(f"Error fetching {url}: {e}")
            attempt += 1
            if attempt < retries:
                print(f"Retrying {url} ({attempt}/{retries})...")
                time.sleep(2)  # Wait before retrying
            else:
                print(f"Failed to fetch {url} after {retries} attempts.")
                return None

# Function to remove header, footer, and extract the remaining content
def extract_main_content(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    # Remove header and footer
    if soup.header:
        soup.header.decompose()
    if soup.footer:
        soup.footer.decompose()

    # Remove page-head element by class
    page_head = soup.find(class_="page-head")
    if page_head:
        page_head.decompose()

    # Remove specific elements by their classes
    specific_classes = [
        "bUqfOz", "hEiKeJ", "gySqrp", "customHeader", "headernavbar", "breadcrumb",
        "customfooter", "content_top", "itr-season-banner", "header-wrapper",
        "common-wrapper", "common-right", "common-left", "container-fluid row",
        "bread_crumbs", "topic", "top-header", "navbar", "nav clearfix",
        "row breadcrumb-outer", "steps px-0", "menu_wrapper", "region region-user-menu",
        "myheadbtnhdr", "top-bar","py-2 text-center"
    ]

    for class_name in specific_classes:
        elements = soup.find_all(class_=class_name)
        for element in elements:
            element.decompose()

    # Get remaining text from the body
    body_content = soup.body.get_text(separator='\n').strip() if soup.body else ""
    return body_content

# Function to chunk text with overlap
def chunk_text(text, chunk_size=100, overlap=20):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = words[i:i + chunk_size]
        chunks.append(' '.join(chunk))
    return chunks

# List of URLs with meaningful names
urls_with_names = {
    "https://www.allahabadhighcourt.in/intro.htm": "Allahabad High Court - - History or About",
    "https://aphc.gov.in/about.html": "Andhra Pradesh High Court - - History or About",
    "https://highcourt.cg.gov.in/history.html": "High Court Of Chhattisgarh - History or About",
    "https://ghconline.gov.in/index.php/profile-2/": "The Gauhati High Court About",
    "https://gujarathighcourt.nic.in/aboutus": "Gujarat High Court - History or About",
    "https://hpkangra.nic.in/h-p-high-court/": "Himachal Pradesh High Court - History or About",
    "https://aphc.gov.in/about.html": "Andhra Pradesh High Court - - History or About",
    "https://karnatakajudiciary.kar.nic.in/newwebsite/aboutus.php": "Karnataka High Court About - History or About",
    "https://mphc.gov.in/history-constitution": "High Court of Madhya Pradesh - History or About",
    "https://hcraj.nic.in/hcraj/history.php": "High Court Rajasthan - History or About",
    "https://ecommitteesci.gov.in/division/high-court-of-madras/": "Madras High Court - History or About",
    "https://meghalayahighcourt.nic.in/history": "Meghalaya High Court - History or About",
    "https://jharkhandhighcourt.nic.in/history.php": "Jharkhand High Court - History or About",
    "https://ecommitteesci.gov.in/division/rajasthan-high-court-2/": "Rajasthan High Court - History or About",
    "https://ecommitteesci.gov.in/division/phc/": "Patna High Court - History or About",
    "https://hcs.gov.in/hcs/JudicialHistory": "Sikkim High Court - History or About",
    "https://ecommitteesci.gov.in/od/division/high-court-of-orissa/": "Orissa High Court - History or About",
    "https://tshc.gov.in/processMenuTypes?id=257": "High Court for the State of Telangana - History or About",
    "https://thc.nic.in/history.html": "Tripura High Court - History or About",
    "https://ecommitteesci.gov.in/" : "e-Committee, Supreme Court of India",
    "https://hcraj.nic.in/hcraj/history.php": "High Court Rajasthan - History or About",
    "https://tshc.gov.in/processMenuTypes?id=257": "High Court for the State of Telangana - History or About",
    "https://thc.nic.in/history.html": "Tripura High Court - History or About",
}

# Initialize an empty list for chunks
high_court_details = []

for url, name in urls_with_names.items():
    try:
        html_content = fetch_html(url)
        if html_content:  # Proceed only if HTML content is successfully fetched
            print(f"Fetched HTML for URL: {url}")  # Debug print

            extracted_text = extract_main_content(html_content)
            if extracted_text:
                print(f"Extracted content from {url} using extract_main_content.")  # Debug print
            else:
                print(f"Can't scrape content from {url} using the <h> and <p> tags")  # Debug print
                soup = BeautifulSoup(html_content, 'html.parser')
                data = soup.find_all(['h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'p'])
                scrap_data = [d.text.strip() for d in data if d.text.strip()]  # Strip and check for empty text
                extracted_text = '\n'.join(scrap_data)

            # Only chunk if there's actual text
            if extracted_text.strip():
                chunked_text = chunk_text(extracted_text, chunk_size=180, overlap=45)
                for i, chunk in enumerate(chunked_text):
                    # Wrap text to ensure it fits within the desired width
                    wrapped_text = textwrap.fill(chunk, width=80)

                    high_court_details.append(f"{name}:\n{wrapped_text}")
            else:
                print(f"No valid content extracted from {url}.")
        else:
            print(f"Failed to fetch content from {url}")  # Debug print

    except Exception as e:
        print(f"Error fetching or processing {url}: {e}")

    time.sleep(1)

# Manually add text data with meaningful names
high_court_texts = {
     "Calcutta High Court :The Calcutta High Court is the oldest High Court in India. It has jurisdiction over the State of West Bengal and the Union Territory of the Andaman and Nicobar Islands. The High Court building's design is based on the Cloth Hall, Ypres, in Belgium. The court has a sanctioned judge strength of 72. The High Court at Calcutta, formerly known as the High Court of Judicature at Fort William, was brought into existence by the Letters Patent dated 14th May, 1862.",
    "Calcutta High Court : The court was opened with Sir Barnes Peacock as its first Chief Justice. Appointed on 2nd February, 1863, Justice Sumboo Nath Pandit was the first Indian to assume office as a Judge of the Calcutta High Court, followed by other legal luminaries like Justice Dwarka Nath Mitter and Justice P.B. Chakravartti, who was the first Indian to become a permanent Chief Justice of the Calcutta High Court.",
      "Manipur High Court:The High Court of Manipur, initially a Permanent Bench of the Gauhati High Court established on January 21, 1972, was formally inaugurated as a full High Court on March 23, 2013, with Hon'ble Shri Justice Abhay Manohar Sapre as its first Chief Justice, after a series of milestones including the laying of the foundation stone on April 30, 2006, the inauguration of the new building on December 3, 2011, and the functioning of the Imphal Bench in the new complex starting April 7, 2012.",
       "Kerala High Court:The Kerala High Court, which leads the judicial administration of Kerala and the Union Territory of Lakshadweep, currently has a sanctioned judge strength of 35 Permanent Judges including the Chief Justice and 12 Additional Judges. The State of Kerala, formed on November 1, 1956, through the merger of Travancore-Cochin and Malabar District, established its High Court on November 1, 1956, with its seat at Ernakulam, following the earlier formation of the High Court of Travancore-Cochin on July 7, 1949. The Kerala High Court is notable for having Justice Anna Chandy as the first woman High Court Judge in India and Justice M. Fathima Beevi as the first woman Judge of the Supreme Court of India." ,
    "Punjab and Haryana High court: Located in Chandigarh, a city on the foothills of the Shivalik range, the Punjab and Haryana High Court serves as the judicial authority for both states. Chandigarh, the capital of Punjab and Haryana and a Union Territory, was established as a new capital after the partition of India, replacing Lahore which went to Pakistan. The Punjab and Haryana High Court, situated in Sector 1 of Chandigarh, has historical roots tracing back to the establishment of the Lahore High Court on March 20, 1919, by Letters Patent under the Government of India Act, 1915. The Lahore High Court was originally vested with appellate and superintending powers and authority, and it had certain original jurisdictions including disciplinary actions and matrimonial matters. It had no ordinary civil jurisdiction but limited original criminal jurisdiction. The High Court could hear appeals from decisions of courts in Punjab, Delhi, and other regions under its superintendence, and it had powers to transfer cases and issue writs in certain circumstances.",
     "Punjab and Haryana High court :This position continued till the Indian Independence Act, 1947 when the dominions of India and Pakistan were created. The High Courts (Punjab) Order, 1947 established a new High Court for the territory of what was then called the East Punjab with effect from 15th August, 1947. ",
     "Punjab and Haryana High court: The India (Adaptation of Existing Indian Laws) Order, 1947 provided that any reference in an existing Indian law to the High Court of Judicature at Lahore, be replaced by a reference to the High Court of East Punjab. The High Court of East Punjab started functioning from Shimla in a building called 'Peterhoff'." ,
    "Delhi High court : The High Court of Delhi was established on 31st October, 1966.Initially, the High Court of Judicature at Lahore, which was established by a Letters Patent dated 21st March, 1919, exercised jurisdiction over the then provinces of the Punjab and Delhi. This position continued till the Indian Independence Act, 1947 when the dominions of India and Pakistan were created.",
     "Delhi High court :1947 when the dominions of India and Pakistan were created.The High Court of Delhi initially exercised jurisdiction not only over the Union Territory of Delhi, but also Himachal Pradesh. The High Court of Delhi had a Himachal Pradesh Bench at Shimla in a building called Ravenswood. The High Court of Delhi continued to exercise jurisdiction over Himachal Pradesh until the State of Himachal Pradesh Act, 1970 was enforced on 25th January, 1971.The High Court of Delhi was established with four Judges. They were Chief Justice K.S.Hegde, Justice I.D.Dua, Justice H.R.Khanna and Justice S.K.Kapur. The sanctioned strength of Judges of this High Court increased from time to time. Presently, the sanctioned strength of Judges of the High Court of Delhi is 45 permanent Judges and 15 Additional Judges.",
     "Uttarakhand High Court:The Uttarakhand High Court was established on November 9, 2000, with its seat in Nainital, having been carved out from the State of Uttar Pradesh. The court operates from a historic building constructed in 1900, originally the old Secretariat, situated in Mallital, Nainital, with picturesque views of Naina Peak. Initially equipped with five courtrooms, the building was later expanded to include additional courtrooms, a Chief Justice Court Block, and a Block of Lawyers’ chambers completed in 2007. The founding Chief Justice was Hon'ble Mr. Justice Ashok A. Desai, who, along with Hon'ble Mr. Justice P.C. Verma and Hon'ble Mr. Justice M.C. Jain, was transferred from the High Court of Allahabad." ,
    "Uttarakhand High Court:The founding Chief Justice was Hon'ble Mr. Justice Ashok A. Desai, who, along with Hon'ble Mr. Justice P.C. Verma and Hon'ble Mr. Justice M.C. Jain, was transferred from the High Court of Allahabad. The court’s sanctioned strength of judges was initially 7, increased to 9 in 2003. Notable former Chief Justices include Hon'ble Mr. Justice S.H. Kapadia, Hon'ble Mr. Justice V.S. Sirpurkar, and Hon'ble Mr. Justice Cyriac Joseph, all of whom were elevated to the Supreme Court of India. Other notable transfers include Hon'ble Mr. Justice P.C. Verma to the Allahabad High Court, and retirements of Hon'ble Mr. Justice Irshad Hussain, Hon'ble Mr. Justice M.M. Ghildiyal, and Hon'ble Mr. Justice Rajesh Tandon.",
     "Jammu and Kashmir and Ladakh : The full-fledged High Court of Judicature for Jammu and Kashmir was established in 1928, prior to which the Maharaja of the state was the final authority in the administration of justice. In 1889, the British Government advised Maharaja Partap Singh to appoint a Council whose Judicial member handled appellate matters, both civil and criminal. Later, this Council was abolished, and a Minister, later designated as Judge of the High Court, was appointed to handle judicial cases. In 1927, a new Constitution was sanctioned by the Maharaja, leading to the formation of a Judicial Ministry, and in 1928, the High Court of Judicature was officially established with Lala Kanwar Sein appointed as the first Chief Justice alongside two puisne judges. The Court sat in both Jammu and Srinagar, marking a pivotal development in the region’s legal structure.",
     "Jammu and Kashmir and Ladakh :The Court sat in both Jammu and Srinagar, marking a pivotal development in the region’s legal structure.In 1928, the High Court of Judicature was officially established with Lala Kanwar Sein appointed as the first Chief Justice alongside two puisne judges, with the Court sitting in both Jammu and Srinagar. The High Court gained significant autonomy through the 1939 Constitution Act of 1996, which gave it superintendence over the District Courts and established a Board of Judicial Advisers akin to the Privy Council for advising the ruler on civil and criminal appeals. This Board was eventually abolished by the 1956 Constitution Act, leading to the creation of a special Supreme Court Bench to settle 17 pending appeals, an unprecedented move with the Supreme Court sitting outside Delhi for the first time.",
     "Jammu and Kashmir and Ladakh :This Board was eventually abolished by the 1956 Constitution Act, leading to the creation of a special Supreme Court Bench to settle 17 pending appeals, an unprecedented move with the Supreme Court sitting outside Delhi for the first time.The Board of Judicial Advisers was eventually abolished by the 1956 Constitution Act, leading to the creation of a special Supreme Court Bench to settle 17 pending appeals, an unprecedented move with the Supreme Court sitting outside Delhi. In 1954, the jurisdiction of the Supreme Court of India was extended to Jammu and Kashmir under the Constitution Application Order, allowing the State High Court to issue writs for enforcing fundamental rights under Article 32(2-A) of the Constitution. This was further consolidated in 1957 by the Jammu and Kashmir Constitution Act, which officially established the High Court of Judicature as an independent judicial body with a sanctioned strength of 17 judges, including 13 Permanent and 4 Additional Judges.",
      "Bombay High Court : he High Court of Bombay, one of the oldest High Courts in the country and a chartered High Court, has appellate jurisdiction over Maharashtra, Goa, Daman & Diu, and Dadra & Nagar Haveli, with its principal seat in Bombay and benches in Aurangabad, Nagpur, and Panaji (Goa). The legal history of Bombay traces back to 1661 when the town and island of Bombay became a British possession as part of the dowry of Portuguese Princess Catherine of Braganza, given to King Charles II upon their marriage. Initially a small fishing village of Kolis, Bombay was transferred to the East India Company by Charles II in 1668 for an annual rent of just 10 Pounds.",
     "Bombay High Court : Bombay was transferred to the East India Company by Charles II in 1668 for an annual rent of just 10 Pounds.In 1668, Bombay was transferred to the East India Company for an annual rent of 10 Pounds, marking the beginning of British legal history in the region. The judicial structure of Bombay began taking shape with the Charter of 1668, under which the administration of justice was placed in the hands of Justices holding court in the Custom Houses of Bombay and Mahim. However, the judicial system remained elementary and closely tied to the executive government. In 1672, Gerald Aungier, the Governor of Surat Factory and often regarded as the \"true founder\" of Bombay, selected George Wilcox as the judge, inaugurating the first British Court of Justice in Bombay with a ceremonial procession through the town.",
     "Bombay High Court : Bombay, selected George Wilcox as the judge, inaugurating the first British Court of Justice in Bombay with a ceremonial procession through the town.In 1672, Gerald Aungier, known for his liberal and impartial approach, appointed George Wilcox as the judge, marking the inauguration of the first British Court of Justice in Bombay. The opening ceremony on August 8, 1672, was a grand affair, featuring a procession through the Bazaar to the Guildhall, with various community representatives, officials, and the judge on horseback. During his address, Aungier emphasized justice without distinction of nationality or religion, declaring that all inhabitants, whether English, Portuguese, Moor, or Gentoo, had equal rights to justice. His passion for fair and even-handed justice set a high standard, though later governors did not follow his noble example.",
}

# Extend the chunks with manual data
high_court_details.extend([textwrap.fill(text, width=80) for text in high_court_texts])




Fetched HTML for URL: https://www.allahabadhighcourt.in/intro.htm
Extracted content from https://www.allahabadhighcourt.in/intro.htm using extract_main_content.
Fetched HTML for URL: https://aphc.gov.in/about.html
Extracted content from https://aphc.gov.in/about.html using extract_main_content.
Fetched HTML for URL: https://highcourt.cg.gov.in/history.html
Extracted content from https://highcourt.cg.gov.in/history.html using extract_main_content.
Fetched HTML for URL: https://ghconline.gov.in/index.php/profile-2/
Extracted content from https://ghconline.gov.in/index.php/profile-2/ using extract_main_content.
Fetched HTML for URL: https://gujarathighcourt.nic.in/aboutus
Extracted content from https://gujarathighcourt.nic.in/aboutus using extract_main_content.
Fetched HTML for URL: https://hpkangra.nic.in/h-p-high-court/
Extracted content from https://hpkangra.nic.in/h-p-high-court/ using extract_main_content.
Fetched HTML for URL: https://karnatakajudiciary.kar.nic.in/newwebsite/aboutus

In [ ]:

for chunk in high_court_details:
    print(chunk)
    print("-" * 80)

Allahabad High Court - - History or About:
Introduction History B y the Indian High Courts Act passed by British Parliament
in 1861, provision was made, not only for the replacement of the Supreme Courts
of Calcutta, Madras and Bombay and for the establishment of High Courts in their
places, but for the establishment of a High Court by Letters Patent in any other
part of Her Majesty’s territories not already included in the jurisdiction of
another High Court. In the year 1866, the High Court of Judicature for the
North-Western Provinces came into existence at Agra under Letters Patent of the
17th March, 1866, replacing the old Sudder Diwanny Adawlat. Sir Walter Morgan,
Barrister-at-Law and Mr. Simpson were appointed the first Chief Justice and the
first Registrar respectively of High Court of North-Western Provinces. The seat
of the High Court for the North-Western Provinces was shifted from Agra to
Allahabad in 1869 and its designation was altered to ‘the High Court of
Judicature at A